# Ensemble Learning to forecast weather

Ensemble has 2 parts: the weak learners and the method of combining these weak learners

There are also 3 types of ensemble learning:
- **Bagging**: each weak learner makes a decision, and the final decision from this ensemble is usually average of decisions or majority vote
- **Boosting**: each weak learner is trained sequentially, and each new model tries to correct the error of the previous one. By focusing on harder and harder to classify problems one weak learner after another, the weighted predictions makes the ensemble much stronger at predicting.
- **Stacking**: Each weak learner is independent, and then a meta-model (a higher level model) learns how to combine these predictions to make the final output. Multi-layer stacking is also possible: feeding first level of base models outputs into a second level and then into a meta-model to create a final output. This can capture a wide variety of patterns in the dataset.

I would try random forest, but I find it hard for forecasting weather as it deals with time.

For this model, I want to use **XGBoost** due to its gradient boosting and its overfitting prevention. Also because it is the most interesting model to me.

### Data Collection

I will try to pull data from the Data.gov.hk website and see if I can get data such as how much rainfall, the highest and lowest temperature, humidity, highest wind speed, the visibility, and the amount of sunshine for each day and try to get like, idk, 5 years worth of data?

Website used:
- Daily Mean Pressure: https://data.gov.hk/en-data/dataset/hk-hko-rss-daily-mean-pressure
- Daily Total Rainfall: https://data.gov.hk/en-data/dataset/hk-hko-rss-daily-total-rainfall
- Daily Max, Mean, Min Temperatures: https://data.gov.hk/en-data/dataset/hk-hko-rss-daily-temperature-info-hko
- Daily Mean Relative Humidity: https://data.gov.hk/en-data/dataset/hk-hko-rss-daily-mean-relative-humidity
- Daily Mean Amount of Clouds: https://data.gov.hk/en-data/dataset/hk-hko-rss-daily-mean-amount-of-cloud

The most recent date in each file is 5/31/2025.

A total of 7 csv files (Temperature has 3). Let's dig in!

### Libraries

In [1]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_squared_error

import matplotlib.pyplot as plt

import plotly.express as px
import plotly.graph_objects as go

### Data Cleaning/Wrangling
A few things to clarify so you don't have to open the CSVs yourself:

**At the top**, it has this message:

日平均雲量(百分比) - 天文台

Daily Mean Amount of Cloud (%) at the Hong Kong Observatory

年/Year,月/Month,日/Day,數值/Value,數據完整性/data Completeness

So I need to get rid of these rows

**At the bottom**, it has this message:

\- "*** 沒有數據/unavailable"

"# 數據不完整/data incomplete"

"C 數據完整/data Complete"

So I need to get rid of these and adjust accordingly, too.

Sorry this markdown chunk looks really disgusting, I am still not the best with markdown right now I need to learn that too.

In [17]:
def clean_data(dat, rename_str):
    dat.columns = ['Year', 'Month', 'Day', 'Value', 'Data Complete']
    dat = dat[:-3]
    # print("-----Before cleaning-----")

    # datDefects = len(dat[dat['Data Complete'] != 'C'])
    # print("Number of faulty rows:",datDefects)
    # print("\n---Summary of Value---")
    # print(dat['Value'].describe())
    # print("\n---Summary of Data Complete---")
    # print(dat['Data Complete'].describe())
    
    # Missing data
    dat = dat[dat['Data Complete'] == 'C']

    dat['Year'] = dat['Year'].astype(int).astype(str)
    dat['Month'] = dat['Month'].astype(int).astype(str)
    dat['Day'] = dat['Day'].astype(int).astype(str)

    dat['Date'] = pd.to_datetime(dat['Year']+"-"+dat['Month']+"-"+dat['Day'], format="%Y-%m-%d")
    dat = dat.set_index('Date')
    
    dat = dat.drop(['Year', 'Month', 'Day', 'Data Complete'], axis=1)

    dat = dat.loc[:, ['Value']]
    dat['Value'] = dat['Value'].astype(float)

    dat = dat.rename(columns = {'Value': rename_str})

    print("-----After cleaning-----")
    print(f"Number of faulty rows: {dat.isna().sum().sum()}")
    # print("---Summary of Data Complete---")
    # print(dat['Data Complete'].describe())
    print("---Dataframe Info---")
    print(dat.info())
    return dat

Cleaning HKO_cloud_amount.csv

In [18]:
cloud = pd.read_csv("data/HKO_cloud_amount.csv", header=2)
cloud = clean_data(cloud, 'Cloud')
cloud.head()

-----After cleaning-----
Number of faulty rows: 0
---Dataframe Info---
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 27910 entries, 1949-01-01 to 2025-05-31
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Cloud   27910 non-null  float64
dtypes: float64(1)
memory usage: 436.1 KB
None


,Cloud
Date,
1949-01-01,68.0
1949-01-02,92.0
1949-01-03,90.0
1949-01-04,94.0
1949-01-05,97.0


Cleaning HKO_max_temp.csv

In [19]:
maxTemp = pd.read_csv("data/HKO_max_temp.csv", header=2)
maxTemp = clean_data(maxTemp, 'MaxTemp')
maxTemp.head()

-----After cleaning-----
Number of faulty rows: 0
---Dataframe Info---
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 49094 entries, 1884-01-01 to 2025-05-31
Data columns (total 1 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   MaxTemp  49094 non-null  float64
dtypes: float64(1)
memory usage: 767.1 KB
None


,MaxTemp
Date,
1884-01-01,15.3
1884-01-02,17.1
1884-01-03,19.6
1884-01-04,23.2
1884-01-05,19.4


Cleaning HKO_mean_temp.csv

In [20]:
meanTemp = pd.read_csv("data/HKO_mean_temp.csv", header=2)
meanTemp = clean_data(meanTemp, 'MeanTemp')
meanTemp.head()

-----After cleaning-----
Number of faulty rows: 0
---Dataframe Info---
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 49003 entries, 1884-04-01 to 2025-05-31
Data columns (total 1 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   MeanTemp  49003 non-null  float64
dtypes: float64(1)
memory usage: 765.7 KB
None


,MeanTemp
Date,
1884-04-01,17.8
1884-04-02,14.6
1884-04-03,14.8
1884-04-04,16.4
1884-04-05,18.3


Cleaning HKO_min_temp.csv

In [21]:
minTemp = pd.read_csv("data/HKO_min_temp.csv", header=2)
minTemp = clean_data(minTemp, 'MinTemp')
minTemp.head()

-----After cleaning-----
Number of faulty rows: 0
---Dataframe Info---
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 49094 entries, 1884-01-01 to 2025-05-31
Data columns (total 1 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   MinTemp  49094 non-null  float64
dtypes: float64(1)
memory usage: 767.1 KB
None


,MinTemp
Date,
1884-01-01,13.7
1884-01-02,14.6
1884-01-03,16.2
1884-01-04,17.0
1884-01-05,13.6


Cleaning HKO_pressure.csv

In [22]:
pressure = pd.read_csv("data/HKO_pressure.csv", header=2)
pressure = clean_data(pressure, 'Pressure')
pressure.head()

-----After cleaning-----
Number of faulty rows: 0
---Dataframe Info---
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 49003 entries, 1884-04-01 to 2025-05-31
Data columns (total 1 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Pressure  49003 non-null  float64
dtypes: float64(1)
memory usage: 765.7 KB
None


,Pressure
Date,
1884-04-01,1014.9
1884-04-02,1018.0
1884-04-03,1014.8
1884-04-04,1011.2
1884-04-05,1011.6


Cleaning HKO_rainfall.csv

In [23]:
rainfall = pd.read_csv("data/HKO_pressure.csv", header=2)
rainfall = clean_data(rainfall, 'Rainfall')
rainfall.head()

-----After cleaning-----
Number of faulty rows: 0
---Dataframe Info---
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 49003 entries, 1884-04-01 to 2025-05-31
Data columns (total 1 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Rainfall  49003 non-null  float64
dtypes: float64(1)
memory usage: 765.7 KB
None


,Rainfall
Date,
1884-04-01,1014.9
1884-04-02,1018.0
1884-04-03,1014.8
1884-04-04,1011.2
1884-04-05,1011.6


Cleaning HKO_relative_humidity.csv

In [24]:
humidity = pd.read_csv("data/HKO_relative_humidity.csv", header=2)
humidity = clean_data(humidity, 'Humidity')
humidity.head()

-----After cleaning-----
Number of faulty rows: 0
---Dataframe Info---
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 28640 entries, 1947-01-01 to 2025-05-31
Data columns (total 1 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Humidity  28640 non-null  float64
dtypes: float64(1)
memory usage: 447.5 KB
None


,Humidity
Date,
1947-01-01,85.0
1947-01-02,86.0
1947-01-03,84.0
1947-01-04,85.0
1947-01-05,85.0


Now we need to combine all of these dataframes into 1 dataframe.

In [34]:
datas = [cloud, maxTemp, meanTemp, minTemp, pressure, rainfall, humidity]

df = pd.merge(datas[0], datas[1], on='Date', how='inner')

for i in range(2, len(datas)):
    df = pd.merge(df, datas[i], on='Date', how='inner')

print(df.shape)
print(df.head())

(27909, 7)
            Cloud  MaxTemp  MeanTemp  MinTemp  Pressure  Rainfall  Humidity
Date                                                                       
1949-01-01   68.0     19.8      17.3     14.7    1013.8    1013.8      81.0
1949-01-02   92.0     19.5      18.3     17.6    1014.8    1014.8      89.0
1949-01-03   90.0     19.0      15.7     12.6    1018.0    1018.0      85.0
1949-01-04   94.0     15.1      13.3     10.7    1020.8    1020.8      75.0
1949-01-05   97.0     15.2      12.3      9.7    1022.2    1022.2      72.0
